# 01. PyBullet Panda 팔 — Google Colab 시각화

**타깃**: Intel Mac (GPU 없음) 유저가 **Google Colab** 에서 Isaac Sim 의 핵심(로봇 로드 · 물리 시뮬 · 센서 렌더)을 GPU 불필요하게 체험.

Colab 은 Ubuntu 22.04 x86_64 리눅스 VM 입니다. PyBullet 의 GUI 창은 뜨지 않지만, **헤드리스(`p.DIRECT`) 모드 + `getCameraImage`** 로 프레임을 수집해서 노트북 안에서 **애니메이션 / 플롯 / 비디오** 로 시각화할 수 있습니다.

![PyBullet](https://pybullet.org/wordpress/wp-content/uploads/2019/03/cropped-pybullet.png)

### 이 노트북에서 배우는 것
1. PyBullet 설치와 기본 세팅
2. Franka Panda URDF 로드
3. 헤드리스 카메라 렌더 → 프레임 수집
4. matplotlib 로 단일 프레임 표시
5. moviepy 로 애니메이션 MP4/GIF 생성
6. 관절 각도 시계열 플롯
7. 간단한 IK(역기구학) 데모

## 1. 의존성 설치

Colab 런타임 한 번 시작할 때 1회만 실행하면 됩니다 (약 30초).

In [ ]:
!pip -q install pybullet==3.2.7 imageio[ffmpeg] numpy matplotlib pillow

In [ ]:
import numpy as np
import pybullet as p
import pybullet_data
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import HTML, Image, display
from base64 import b64encode

plt.rcParams['figure.dpi'] = 100

## 2. 헬퍼: 시뮬레이터 초기화

- `p.DIRECT` = 헤드리스. Colab 에 물리 GUI 창은 없으니 이걸 써야 합니다.
- `pybullet_data.getDataPath()` 경로에 Panda, KUKA, Husky 등 예제 URDF 가 들어 있습니다.

In [ ]:
def init_sim():
    p.resetSimulation() if p.getConnectionInfo()['isConnected'] else None
    try:
        p.disconnect()
    except Exception:
        pass
    client = p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.loadURDF('plane.urdf')
    robot = p.loadURDF('franka_panda/panda.urdf', useFixedBase=True)
    return robot


def capture_frame(width=480, height=360, eye=(1.2, 1.2, 1.0), target=(0, 0, 0.3)):
    view = p.computeViewMatrix(cameraEyePosition=eye,
                               cameraTargetPosition=target,
                               cameraUpVector=[0, 0, 1])
    proj = p.computeProjectionMatrixFOV(fov=60, aspect=width/height,
                                        nearVal=0.05, farVal=10)
    _, _, rgba, _, _ = p.getCameraImage(width, height, view, proj,
                                         renderer=p.ER_TINY_RENDERER)
    rgb = np.reshape(rgba, (height, width, 4))[:, :, :3].astype(np.uint8)
    return rgb

## 3. Hello Panda — 단일 프레임 스냅샷

In [ ]:
robot = init_sim()

# 안정화 1초
for _ in range(240):
    p.stepSimulation()

img = capture_frame()

plt.figure(figsize=(6, 4.5))
plt.imshow(img)
plt.axis('off')
plt.title('Franka Panda — initial pose')
plt.show()

## 4. 관절 구조 한눈에

PyBullet 에서 Panda 는 총 9개 관절이 정의돼 있습니다 (팔 7개 + 손가락 2개). 각 관절 정보를 테이블로 뽑아봅시다.

In [ ]:
import pandas as pd

rows = []
for i in range(p.getNumJoints(robot)):
    info = p.getJointInfo(robot, i)
    rows.append({
        'idx': i,
        'name': info[1].decode(),
        'type': ['REV', 'PRIS', 'SPHER', 'PLANAR', 'FIXED'][info[2]],
        'lower': round(info[8], 3),
        'upper': round(info[9], 3),
        'max_force': round(info[10], 1),
        'max_velocity': round(info[11], 2),
    })
pd.DataFrame(rows)

## 5. 관절 궤적 애니메이션 (GIF)

7 개 팔 관절에 사인파 궤적을 주고 프레임을 모아 GIF 로 만듭니다. Colab 안에서 바로 재생됩니다.

In [ ]:
robot = init_sim()
n_arm = 7
frames = []
traj_history = []

duration_sec = 4.0
dt = 1.0 / 240.0
capture_every = 8   # 30 FPS 정도

for step in range(int(duration_sec / dt)):
    t = step * dt
    target = [
        0.7 * np.sin(1.5 * t),
        -0.5 + 0.3 * np.sin(1.0 * t),
        0.4 * np.sin(2.0 * t),
        -2.0 + 0.4 * np.sin(0.8 * t),
        0.3 * np.sin(1.2 * t),
        1.2 + 0.3 * np.sin(1.5 * t),
        0.5 * np.sin(1.0 * t),
    ]
    for i, ang in enumerate(target):
        p.setJointMotorControl2(robot, i, p.POSITION_CONTROL,
                                 targetPosition=ang, force=87)
    p.stepSimulation()

    traj_history.append((t, [p.getJointState(robot, i)[0] for i in range(n_arm)]))
    if step % capture_every == 0:
        frames.append(capture_frame(width=400, height=300))

print(f'수집 프레임: {len(frames)}, 궤적 샘플: {len(traj_history)}')

In [ ]:
gif_path = 'panda.gif'
imageio.mimsave(gif_path, frames, fps=30, loop=0)

with open(gif_path, 'rb') as f:
    display(Image(data=f.read(), format='gif'))

## 6. 관절 각도 시계열 플롯

같은 궤적 데이터를 matplotlib 으로 그려 보면, 각 관절이 서로 다른 주파수/위상의 사인파를 어떻게 따라가는지 한눈에 보입니다.

In [ ]:
times = np.array([t for t, _ in traj_history])
joints = np.array([q for _, q in traj_history])

fig, ax = plt.subplots(figsize=(9, 4))
for i in range(joints.shape[1]):
    ax.plot(times, joints[:, i], label=f'joint{i+1}')
ax.set_xlabel('time [s]')
ax.set_ylabel('angle [rad]')
ax.set_title('Panda arm joint trajectories')
ax.legend(ncol=4, loc='upper right', fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## 7. IK(역기구학)로 엔드 이펙터가 목표점 찍기

`calculateInverseKinematics` 를 쓰면 **'end-effector 를 이 좌표로 보내고 싶다'** 고만 말해도 알아서 7 개 관절 각을 찾아 줍니다. Isaac Sim 이나 MoveIt2 의 motion planning 사용감과 같습니다.

In [ ]:
robot = init_sim()
ee_link = 11   # Panda 의 end-effector link 인덱스

# 가고 싶은 점 3개
targets = [
    [0.5,  0.0, 0.5],
    [0.4,  0.3, 0.6],
    [0.3, -0.3, 0.4],
]

frames2 = []
for tgt in targets:
    q = p.calculateInverseKinematics(robot, ee_link, tgt)
    for i in range(7):
        p.setJointMotorControl2(robot, i, p.POSITION_CONTROL,
                                 targetPosition=q[i], force=87)
    for _ in range(240):   # 1초 동안 수렴
        p.stepSimulation()
        frames2.append(capture_frame(320, 240))

gif_path2 = 'panda_ik.gif'
imageio.mimsave(gif_path2, frames2[::8], fps=30, loop=0)
with open(gif_path2, 'rb') as f:
    display(Image(data=f.read(), format='gif'))

## 8. 마무리

- Colab 기본 런타임은 **Ubuntu 22.04 x86_64 CPU 머신** 입니다 (T4 GPU 는 런타임 변경 시 선택 가능). PyBullet 은 CPU 로도 충분히 빠르고, 위의 예제 전부 Mac 에서 네이티브로도 동일하게 돌아갑니다 (`p.connect(p.GUI)` 로 바꾸기만 하면).
- 이 노트북이 보여주는 것은 Isaac Sim 의 3개 핵심 기능의 **축소판**:
  1. 로봇 URDF 로드 → `loadURDF`
  2. 물리 시뮬레이션 루프 → `stepSimulation`
  3. 센서 렌더링 → `getCameraImage`
- 다음 노트북에서는 같은 개념을 **ROS 2 토픽** 으로 감싸 봅니다.